In [12]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys

sys.path.insert(1, "/content/drive/MyDrive/modules")


import subscriptionManager as subMan

sys.path.insert(1, "/content/drive/MyDrive")

import feedbackManager as feeMan




Mounted at /content/drive


# Game Management System
**Overview**

This system uses ipywidgets and matplotlib to create an interactive and interesting user experience, making it easier to manage the game system.

**Functionalities**

*1.Rent Game*

User can enter a game ID along with their own details and rent a game from the database.

*2.Return Game*

Allows the user to return a rented game and give feedback on it. The program will ask for the date, the game ID, a rating, and optional comments.

*3.Board Game/Video Game Search*

The user can choose between serching for a board game or a video game in the main menu. From there, both buttons do the same thing; allowing the user to choose between searching by title or genre and then seraching for the chosen game and displaying all the information on the searched game.

*4.Booking Sessions*

The user can book sessions at two certain times from a drop-down menu. If all field are entered correctly, their booking will be accepted and processed.

*5.Pruning*

The user can choose to prune either the video games or board games. The result will be a table of poorly peforming games, along with a bar chart of overall game ratings.

*6.Quit*

Allows the user to exit the system.

Each of these features are accesbile through a button on the main menu. Enjoy!






In [13]:
#Wrote by: F529108
#Last edited: 09/01/2026
import os
os.chdir("/content/drive/MyDrive")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
menu = widgets.Output()

run=True

filepath = "/content/drive/MyDrive"


#---------------------------------
# Function Definitions
#---------------------------------

#-----------------------
#Function - gameReturn
#-----------------------
def return_game(b):
  """
  Allows the user to return a game and leave feedback.
  """
  with menu:
    #create menu for return section
    menu.clear_output()
    date_input = widgets.Text(description='Enter Date:', placeholder='****-**-**')
    gamid_input = widgets.Text(description='Enter Game ID:', style={'description_width': 'initial'})
    rating_input = widgets.Text(description='Enter Star Rating:', style={'description_width': 'initial'})
    comment_input = widgets.Text(description='Enter Comments:', style={'description_width': 'initial'})
    submit_btn = widgets.Button(description='Submit')
    action_output = widgets.Output()
    ui_layout = widgets.VBox([date_input, gamid_input, rating_input, comment_input, submit_btn, action_output])

    def on_submit_click(b):
      """
      Submits the data and updates the feedback.
      """
      with action_output:
        action_output.clear_output()
        found = False
        valid_input = False
        date = date_input.value
        gamid = gamid_input.value
        ratingstr = rating_input.value
        try:
          rating = int(ratingstr)
          valid_input = True
        except ValueError:
          print('Error: Rating must be a whole number\n')
        if valid_input == True:
          comments = comment_input.value
          feeMan.add_feedback(gamid, rating, comments, 'Game_Feedback.txt')
          print('Feedback Recieved!\n')
          #initialise matrix
          rMatrix=[]
          #iterate through databse to create a matrix of the database to work with
          with open('Rental.txt'.format(path=filepath), 'r') as f:
              f.readline()
              for line in f:
                  s = line.strip()
                  rMatrix.append(list(s.split(",")))
          for i in range(len(rMatrix)):
              if rMatrix[i][0] == gamid and rMatrix[i][2] == "":
                  rMatrix[i][2] = date
                  found = True
          if found == False:
            print('Error - Could not return\n')

          with open('Rental.txt', 'w') as f:
              f.write("GameID,Rental Date,Return Date,Rental Customer ID\n")
              for line in rMatrix:
                f.write(",".join(line)+"\n")
        else:
          print('Could not complete return\n')
    #link button to event
    submit_btn.on_click(on_submit_click)
    display(ui_layout)



#-----------------------
#New Function - gameRent
#-----------------------
def rent_game(b):
  """
  Used so the user can rent out games.

  Updates records accordingly.
  """
  with menu:
    #create menu area for renting
    menu.clear_output()
    cusid_input = widgets.Text(description='Customer ID:', placeholder='Enter ID')
    gamid_input = widgets.Text(description='Game ID:', placeholder='Enter Game ID')
    date_input = widgets.Text(description='Date:', placeholder='****-**-**')
    sub_button = widgets.Button(description="Submit")
    action_output = widgets.Output()
    #create a container to display the submenu as
    ui_container = widgets.VBox([
    cusid_input,
    gamid_input,
    date_input,
    sub_button,
    action_output
    ])

    def on_sub_click(b):
      """
      Submits and runs the renting program using the data collected.
      """

      availability = True
      cus_rented = 0


      with action_output:
            action_output.clear_output()
            cusid = cusid_input.value
            gamid = gamid_input.value
            current_date = date_input.value
            subscriptions = subMan.load_subscriptions()
            validation = subMan.check_subscription(cusid, subscriptions)

            if cusid and gamid and validation == True:
              #initialise matrix
              rMatrix = []
              #iterate through database, appending data to the matrix
              with open('Rental.txt'.format(path=filepath), 'r') as f:
                  f.readline()
                  for line in f:
                      s = line.strip()
                      rMatrix.append(list(s.split(",")))
              for i in range(len(rMatrix)):
                  if rMatrix[i][2] == "" and rMatrix[i][0] == gamid:
                      availability = False
              for i in range(len(rMatrix)):
                if rMatrix[i][3] == cusid and rMatrix[i][2] == "":
                  cus_rented = cus_rented +1
              siMatrix = []
              with open('Subscription_Info.txt'.format(path=filepath), 'r') as f:
                  f.readline()
                  for line in f:
                      s = line.strip()
                      siMatrix.append(list(s.split(",")))
              sub_type = ""
              for i in range(len(siMatrix)):
                if siMatrix[i][0] == cusid:

                  sub_type = siMatrix[i][1]

              if availability == True and (cus_rented+1) <= subMan.get_rental_limit(sub_type) and current_date != "":



                with open('Rental.txt', 'a') as f:
                  f.write(",".join([gamid, current_date, "", cusid]) + "\n")


                print('Rental Completed!\n')
                print(rMatrix)
              elif availability == False and (cus_rented+1) <= subMan.get_rental_limit(sub_type):
                print('Invalid - Game is not available\n')
              elif availability == True and (cus_rented+1) > subMan.get_rental_limit(sub_type):
                print('Invalid - Customer has met max subscription limit\n')
              else:
                print('Invalid - Cannot complete rental\n')

            else:
              print('Invalid input - Incorrect ID(s)')

    #link button to event handler
    sub_button.on_click(on_sub_click)
    display(ui_container)


#-------------------------
#New Function - gameSearch
#-------------------------
def search_bgame(b):
  """
  Used to search for board games based on title or genre.
  """
  with menu:
    #create menu for seraching for board games(s)
    menu.clear_output()
    label_style = {'description_width': 'initial'}
    title_btn = widgets.Button(description='Title')
    genre_btn = widgets.Button(description='Genre')
    result_output = widgets.Output()
    display(widgets.HBox([title_btn, genre_btn]))
    display(result_output)

    #Buttons for running the program under title or genre
    def on_title_click(b):
      with result_output:
        result_output.clear_output()
        title_input = widgets.Text(description='Enter Title:')
        search_button = widgets.Button(description='Search')
        action_output = widgets.Output()
        display(widgets.VBox([title_input, search_button, action_output]))

        def on_search_click(b):
          """
          Completes title search.
          """
          with action_output:
            action_output.clear_output()
            game_found = False
            title=title_input.value.lower()
            #populate matrix
            bgMatrix = []
            with open('Board_Game_Info.txt', 'r') as f:
                f.readline()
                for line in f:
                    s = line.strip()
                    bgMatrix.append(list(s.split(",")))

            for i in range(len(bgMatrix)):
                if bgMatrix[i][1].lower() == title:
                    print(bgMatrix[i])
                    game_found = True
            if game_found == False:
              print('Could not find game.\n')
            else:
              print('\n')
        search_button.on_click(on_search_click)

    # Same for genre.
    def on_genre_click(b):
      with result_output:
        result_output.clear_output()
        genre_input = widgets.Text(description='Enter Genre:')
        search_button2 = widgets.Button(description='Search')
        action_output2 = widgets.Output()
        display(widgets.VBox([genre_input, search_button2, action_output2]))

        def on_search2_click(b):
              with action_output2:
                action_output2.clear_output()
                game_found = False
                genre= genre_input.value.lower()
                bgMatrix = []
                with open('Board_Game_Info.txt', 'r') as f:
                    f.readline()
                    for line in f:
                        s = line.strip()
                        bgMatrix.append(list(s.split(",")))

                for i in range(len(bgMatrix)):
                    if bgMatrix[i][3].lower() == genre:
                        print(bgMatrix[i])
                        game_found = True
                if game_found == False:
                  print('Could not find game.\n')
                else:
                  print('\n')
        search_button2.on_click(on_search2_click)


    #link buttons to events
    title_btn.on_click(on_title_click)
    genre_btn.on_click(on_genre_click)

#-------------------------
#New Function - gameSearch
#-------------------------
def search_vgame(b):
  """
  Used to search for video games based on title or genre.
  """
  with menu:
    #create menu for video game searching
    menu.clear_output()
    label_style = {'description_width': 'initial'}
    title_btn = widgets.Button(description='Title')
    genre_btn = widgets.Button(description='Genre')
    result_output = widgets.Output()
    display(widgets.HBox([title_btn, genre_btn]))
    display(result_output)

    # Buttons for running the program under title or genre.
    def on_title_click(b):
      with result_output:
        result_output.clear_output()
        title_input = widgets.Text(description='Enter Title:')
        search_button = widgets.Button(description='Search')
        action_output = widgets.Output()
        display(widgets.VBox([title_input, search_button, action_output]))

        def on_search_click(b):
          """
          Completes title search.
          """
          with action_output:
            action_output.clear_output()
            game_found = False
            title=title_input.value.lower()
            #populate matrix
            vgMatrix = []
            with open('Video_Game_Info.txt', 'r') as f:
                f.readline()
                for line in f:
                    s = line.strip()
                    vgMatrix.append(list(s.split(",")))

            for i in range(len(vgMatrix)):
                if vgMatrix[i][1].lower() == title:
                    print(vgMatrix[i])
                    game_found = True
            if game_found == False:
              print('Could not find game.\n')
            else:
              print('\n')
        search_button.on_click(on_search_click)

    #Same for genre.
    def on_genre_click(b):
      with result_output:
        result_output.clear_output()
        genre_input = widgets.Text(description='Genre:')
        search_button2 = widgets.Button(description='Search')
        action_output2 = widgets.Output()
        display(widgets.VBox([genre_input, search_button2, action_output2]))

        def on_search2_click(b):
              with action_output2:
                action_output2.clear_output()
                game_found = False
                genre= genre_input.value.lower()
                vgMatrix = []
                with open('Video_Game_Info.txt', 'r') as f:
                    f.readline()
                    for line in f:
                        s = line.strip()
                        vgMatrix.append(list(s.split(",")))

                for i in range(len(vgMatrix)):
                    if vgMatrix[i][3].lower() == genre:
                        print(vgMatrix[i])
                        game_found = True
                if game_found == False:
                  print('Could not find game.\n')
                else:
                  print('\n')
        search_button2.on_click(on_search2_click)


    #link buttons to events
    title_btn.on_click(on_title_click)
    genre_btn.on_click(on_genre_click)

#-----------------------
#New Function - booking
#-----------------------
def booking(b):
  """
  Used to allow the user to book in-house sessions at certain times.
  """
  with menu:
    #create menu for booking
    menu.clear_output()


    valid_input = False
    cusid_input = widgets.Text(description='Enter Customer ID:', style={'description_width': 'initial'})
    date_input = widgets.Text(description='Enter Date:', placeholder='****-**-**', style={'description_width': 'initial'})
    guest_input = widgets.Text(description='No. of Guests:', placeholder='Max:3', style={'description_width': 'initial'})
    slot_input = widgets.Dropdown(options=['2PM - 6PM', '6PM - 10PM'], value = '2PM - 6PM', description='Time Slot')
    book_btn = widgets.Button(description='Book Now')
    result_output = widgets.Output()
    ui_layout = widgets.VBox([cusid_input, date_input, guest_input, slot_input, book_btn, result_output])

    def on_book_click(b):
      """
      Completes the booking after the data is gathered.
      """
      with result_output:
        result_output.clear_output()
        attended = 0
        valid_input = False
        cusid = cusid_input.value
        date = date_input.value
        slot = slot_input.value
        try:
          guests = int(guest_input.value)
          valid_input = True
        except ValueError:
          print('Error: Guests must be a whole number')
        if valid_input == True and guests <= 3 and date != "":

          if slot == '2PM - 6PM':
            slottime = '2pm'
          else:
            slottime = '6pm'

          subscriptions = subMan.load_subscriptions()
          validation = subMan.check_subscription(cusid, subscriptions)
          if validation == True:
            #populate matrix
            bMatrix = []
            with open('Booking.txt'.format(path=filepath), 'r') as f:
                f.readline()
                for line in f:
                    s = line.strip()
                    bMatrix.append(list(s.split(",")))
            for i in range(len(bMatrix)):
                if bMatrix[i][1] == date and bMatrix[i][2] == slottime:
                  attended = attended + 1
                  attended = attended + int((bMatrix[i][3]))
            if  (attended + (guests+1)) <= 50:
              print('Slot Booked!\n')
              with open('Booking.txt', 'a') as f:
                f.write(",".join([cusid, date, slottime, str(guests)]) + "\n")

          else:
            print('Invalid input - Cannot find customer ID')
        else:
          print('Could not complete booking\n')
    #link button to event
    book_btn.on_click(on_book_click)
    display(ui_layout)

#-------------------------------
#New Function - inventoryPruning
#-------------------------------
def pruning(b):
  """
  Goes through the database to find the games that have been rented the least.
  It also displays a bar chart of the games rating.
  """
  with menu:
    #create menu for pruning
    menu.clear_output()
    label_style = {'description_width': 'initial'}
    video_btn = widgets.Button(description='Video Games')
    board_btn = widgets.Button(description='Board Games')
    result_output = widgets.Output()
    display(widgets.HBox([video_btn, board_btn]))
    display(result_output)

    def on_video_click(b):
      """
      Completes the Pruning of Video Games.
      """
      with result_output:
        result_output.clear_output()
        #populate matrix
        vgMatrix = []
        with open('Video_Game_Info.txt'.format(path=filepath), 'r') as f:
            f.readline()
            for line in f:
                s = line.strip()
                vgMatrix.append(list(s.split(",")))
        for i in range(len(vgMatrix)):
          gamid = vgMatrix[i][0]
          gamname = vgMatrix[i][1]
          total=0
          rMatrix = []
          with open('Rental.txt'.format(path=filepath), 'r') as f1:
            f1.readline()
            for line in f1:
              s= line.strip()
              rMatrix.append(list(s.split(",")))
          for i in range(len(rMatrix)):
            if gamid == rMatrix[i][0]:
              total = total+1
          if total < 3:
            print('Game ID '+str(gamname)+ ' has only been rented '+str(total)+ ' times!')
        print('End of Pruning\n')
        feedback_chart()

    def on_board_click(b):
      """
      Completes the pruning of board games.
      """
      with result_output:
        result_output.clear_output()
        #populate matrix
        bgMatrix = []
        with open('Board_Game_Info.txt'.format(path=filepath), 'r') as f:
            f.readline()
            for line in f:
                s = line.strip()
                bgMatrix.append(list(s.split(",")))
        for i in range(len(bgMatrix)):
          gamid = bgMatrix[i][0]
          gamname = bgMatrix[i][1]
          total=0
          rMatrix = []
          with open('Rental.txt'.format(path=filepath), 'r') as f1:
            f1.readline()
            for line in f1:
              s= line.strip()
              rMatrix.append(list(s.split(",")))
          for i in range(len(rMatrix)):
            if gamid == rMatrix[i][0]:
              total = total+1
          if total < 3:
            print('Game ID '+str(gamname)+ ' has only been rented '+str(total)+ ' times!')

        print('End of Pruning\n')
        feedback_chart()

    def feedback_chart():
      """
      Displays the bar chart of game ratings
      """
      name=[]
      rating=[]
      #populate matrix
      gfMatrix=[]
      with open('Game_Feedback.txt', 'r') as f:
          f.readline()
          for line in f:
              s=line.strip()
              gfMatrix.append(list(s.split(",")))
      #populate game names and ratings
      for i in range(len(gfMatrix)):
        name.append(gfMatrix[i][0])
        rating.append(gfMatrix[i][1])

      data = sorted(zip(name, rating), key=lambda x: x[1], reverse=False)
      sorted_name, sorted_rating = zip(*data)

      plt.figure(figsize=(10,5))
      plt.bar(sorted_name, sorted_rating)
      plt.xlabel('Game ID')
      plt.ylabel('Rating')
      plt.title('Bar Chart of Game Ratings')
      plt.xticks(rotation=45, ha='right')
      plt.show()
    #link buttons to events
    video_btn.on_click(on_video_click)
    board_btn.on_click(on_board_click)

#----------------------
#New Function - quit
#----------------------
def quit(b):
  """
  Used to end the program and disable the menu
  """
  menu.clear_output()
  with menu:
    #disables buttons
    print('Goodbye!')
    btn_Rent_Game.disabled = True
    btn_Quit.disabled = True
    btn_Return_Game.disabled = True
    btn_search_bgame.disabled = True
    btn_search_vgame.disabled = True
    btn_Book_Sessions.disabled = True
    btn_Prune_Game.disabled = True





In [14]:
#---------------------------------
# Title
#---------------------------------

print('==========================')
print('==GAME MANAGEMENT SYSTEM==')
print('==========================')

#----------------------------------------
# Create buttons for menu
#----------------------------------------

btn_Rent_Game = widgets.Button(description='Rent Game', button_style='info')
btn_search_vgame = widgets.Button(description='Video Game Search', style={'description_width': 'initial'}, button_style='info')
btn_search_bgame = widgets.Button(description='Board Game Search', style={'description_width': 'initial'}, button_style='info')
btn_Book_Sessions = widgets.Button(description='Book in-house group sessions', layout=widgets.Layout(width='auto'), button_style='info')
btn_Return_Game = widgets.Button(description='Return a game', button_style='info')
btn_Prune_Game = widgets.Button(description='Prune', button_style='info')
btn_Quit = widgets.Button(description='Quit', button_style='info')

#----------------------------------------
# Link buttons to event handler
#----------------------------------------

btn_Rent_Game.on_click(rent_game)
btn_Quit.on_click(quit)
btn_Return_Game.on_click(return_game)
btn_search_bgame.on_click(search_bgame)
btn_search_vgame.on_click(search_vgame)
btn_Book_Sessions.on_click(booking)
btn_Prune_Game.on_click(pruning)

#-------------------------------------------
# Display menu GUI
#-------------------------------------------

display(btn_Rent_Game, btn_Return_Game, btn_search_bgame, btn_search_vgame, btn_Book_Sessions,btn_Prune_Game, btn_Quit)
display(menu)

==GAME MANAGEMENT SYSTEM==


Button(button_style='info', description='Rent Game', style=ButtonStyle())

Button(button_style='info', description='Return a game', style=ButtonStyle())

Button(button_style='info', description='Board Game Search', style=ButtonStyle())

Button(button_style='info', description='Video Game Search', style=ButtonStyle())

Button(button_style='info', description='Book in-house group sessions', layout=Layout(width='auto'), style=But…

Button(button_style='info', description='Prune', style=ButtonStyle())

Button(button_style='info', description='Quit', style=ButtonStyle())

Output()